In [ ]:
import numpy as np
import sleap
from sleap.info.write_tracking_h5 import get_occupancy_and_points_matrices

# Print some info:
sleap.versions()
sleap.system_summary()

### Load labels and remove empty frames

In [ ]:
labels = sleap.load_file("predictions_for_moseq/single_animal_CameraNestTop_topdown_top.single_instance.n=600/CameraNestNorthTop_2023-06-28T15-00-00.slp")
labels.remove_empty_frames()

### Find start and end frames of bouts where the mouse is in frame

In [ ]:
# Get occupancy matrix
matrices = get_occupancy_and_points_matrices(labels,all_frames=True)
occupancy_matrix = np.squeeze(matrices[0])

# Find the frames where occupancy_matrix is 1 (i.e. mouse is present)
frames = np.where(occupancy_matrix == 1)[0]

# Split the frames array to get subarrays of consecutive frames
split_frames = np.split(frames, np.where(np.diff(frames) != 1)[0] + 1)

# Filter the split_frames subarrays to remove those with less than 10 frames (arbitrary number, can be changed)
filtered_split_frames = [subarray for subarray in split_frames if len(subarray) > 10]

# Only keep the start and end frame of each subarray
start_end_frame_pairs = [(subarr[0], subarr[-1]) for subarr in filtered_split_frames]

### Split video and h5 files

In [ ]:
video = labels.video.backend.filename

for i, pair in enumerate(start_end_frame_pairs):
    print("Saving trimmed files number", i+1, "of", len(start_end_frame_pairs), "...")
          
    # Create a sub video from the original video using the start and end frames
    sub_video = video[0:-4] + "_trimmed" + f"_{i+1}" + video[-4:]
    start_frame = pair[0]
    end_frame = pair[1]
    %system ffmpeg -i {video} -vf "select=between(n\,{start_frame}\,{end_frame})" -q:v 0 {sub_video}"

    # Create sub sleap and h5 files from the original sleap file using the start and end frames
    inds = np.arange(start_frame, end_frame+1)
    sub_labels = labels.extract(inds, copy=True)
    sub_labels.video.backend.filename = sub_video
    # Optionally save the sub sleap file (not necessary for MoSeq, can be commented out)
    # sub_sleap_file = video[0:-4] + "_trimmed" + f"_{i+1}" + ".slp"
    # sleap.Labels.save_file(sub_labels, sub_sleap_file)
    # Save the corresponding sub h5 file
    sub_h5_file = video[0:-4] + "_trimmed" + f"_{i+1}" + ".h5"
    sleap.Labels.save_file(sub_labels, sub_h5_file)